# LLaMA — Open and Efficient Foundation Language Models (2023)

_Papers · Transformers & LLMs_

**Train smaller models on far more tokens, on public data only, with three clean architecture tweaks.**

---

This notebook is a practice scaffold for the **LLaMA — Open and Efficient Foundation Language Models (2023)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch
import torch.nn as nn

# ---------------------------------------------------------------------------
# RMSNorm: LayerNorm WITHOUT the mean-subtraction. Divide by the root mean
# square only, then apply a per-entry gain. (Standard definition, Zhang &
# Sennrich 2019 -- the LLaMA paper cites it but does not print the formula.)
# ---------------------------------------------------------------------------
def rms_norm(x, gain, eps=1e-6):
    rms = torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + eps)
    return x / rms * gain

# The toy vector from the worked example.
x    = torch.tensor([1.0, 2.0, 3.0, 4.0])
gain = torch.ones(4)                      # g_i = 1, so we see the raw effect

# (1) RMSNorm -- rescale only, no re-centering.
y_rms = rms_norm(x, gain)

# (2) PyTorch's own LayerNorm -- re-centers (subtracts the mean) then rescales.
ln = nn.LayerNorm(4, elementwise_affine=False)   # no learned scale/shift
y_ln = ln(x)

torch.set_printoptions(precision=4)
print("x          =", x.tolist())
print("RMSNorm(x) =", [round(v, 4) for v in y_rms.tolist()])
print("LayerNorm  =", [round(v, 4) for v in y_ln.tolist()])
print("mean mu    =", x.mean().item(), "  <- the offset only LayerNorm removes")
# x          = [1.0, 2.0, 3.0, 4.0]
# RMSNorm(x) = [0.3651, 0.7303, 1.0954, 1.4606]   <- all positive: rescale only
# LayerNorm  = [-1.3416, -0.4472, 0.4472, 1.3416] <- centered at 0: mean removed
# mean mu    = 2.5   <- the offset only LayerNorm removes
# The whole gap between the two outputs is that mu = 2.5. RMSNorm leaves it in;
# LayerNorm subtracts it. This is OUR illustration, not a number from the paper.

## Visualize the data & results

_RMSNorm is LayerNorm without the mean-subtraction. On one toy vector, how do their outputs differ -- and is the whole difference just the mean that RMSNorm leaves in?_

In [ ]:
import torch
import torch.nn as nn

def rms_norm(x, gain, eps=1e-6):
    rms = torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + eps)
    return x / rms * gain

x    = torch.tensor([1.0, 2.0, 3.0, 4.0])
gain = torch.ones(4)

y_rms = rms_norm(x, gain)
y_ln  = nn.LayerNorm(4, elementwise_affine=False)(x)

print("RMS(x)     =", round(torch.sqrt(x.pow(2).mean()).item(), 4))
print("RMSNorm(x) =", [round(v, 4) for v in y_rms.tolist()])
print("LayerNorm  =", [round(v, 4) for v in y_ln.tolist()])
print("mean mu    =", x.mean().item())
# RMS(x)     = 2.7386
# RMSNorm(x) = [0.3651, 0.7303, 1.0954, 1.4606]
# LayerNorm  = [-1.3416, -0.4472, 0.4472, 1.3416]
# mean mu    = 2.5
# The whole gap is the mean mu=2.5 that only LayerNorm removes. OUR illustration,
# not the paper's numbers.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Why train the smaller model longer? Suppose model A has 65 billion parameters and model B has
            13 billion, and (hypothetically) they reach the same quality &mdash; B only by being trained on many
            more tokens. You will deploy the winner to serve 1 billion requests. (a) Which model is cheaper to
            serve, and roughly by what factor per request? (b) Why might B still be the right choice even
            though it cost more FLOPs to train?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Inference cost per request scales with model size (number of parameters used per forward pass). — _Each generated token runs the whole network once; a bigger network does proportionally more arithmetic per token._
- Compare sizes: 65B vs 13B is a ratio of $65/13 = 5$. So model B is about $5\times$ cheaper per request. — _Five times fewer parameters means roughly five times less compute per forward pass._
- Multiply by request volume: across 1 billion requests, that $5\times$ saving is paid 1 billion times, while the extra training cost is paid once. — _This is exactly LLaMA's argument: the one-time training premium is dwarfed by the repeated inference savings at deployment scale._

**Answer:** (a) The 13B model is about $65/13 = 5\times$ cheaper to serve per request. (b) Because that
                 $5\times$ saving is paid on every one of the billion requests, while B's higher training FLOPs are a
                 one-time cost. That is LLaMA's thesis (Introduction): "a smaller one trained longer will ultimately
                 be cheaper at inference." Optimize the inference budget, not just the training budget.

</details>

**Problem 2.** RMSNorm by hand. Take the vector $x = (3, 4)$ with gain $g = 1$ and $\epsilon$ ignored.
            (a) Compute $\mathrm{RMSNorm}(x)$. (b) Compute $\mathrm{LayerNorm}(x)$ (no learned shift). (c) State in
            one sentence why the two differ here.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- RMSNorm: sum of squares $= 9 + 16 = 25$, mean of squares $= 25/2 = 12.5$, $\mathrm{RMS} = \sqrt{12.5} \approx 3.5355$. Divide: $(3/3.5355,\, 4/3.5355) \approx (0.8485,\, 1.1314)$. — _RMSNorm divides each entry by the root mean square and never subtracts the mean, so both stay positive._
- LayerNorm: mean $\mu = 3.5$, centered $(-0.5, 0.5)$, variance $= (0.25 + 0.25)/2 = 0.25$, $\sigma = 0.5$. Divide: $(-1, 1)$. — _LayerNorm subtracts the mean first, which here flips the smaller entry negative._
- Compare: RMSNorm gave $(0.85, 1.13)$, LayerNorm gave $(-1, 1)$. The difference is the mean $\mu = 3.5$ that LayerNorm removed. — _The only structural difference between the two is LayerNorm's re-centering step; with a nonzero mean it changes the output._

**Answer:** (a) $\mathrm{RMSNorm}(x) \approx (0.8485,\, 1.1314)$. (b) $\mathrm{LayerNorm}(x) = (-1,\, 1)$.
                 (c) They differ because LayerNorm subtracts the mean $\mu = 3.5$ (re-centering) while RMSNorm does
                 not; the nonzero mean is exactly what RMSNorm leaves in. With a mean-zero input the two would be
                 proportional.

</details>

**Problem 3.** Ablation: put the mean-subtraction back. You modify a LLaMA-style block by switching RMSNorm
            back to LayerNorm, changing nothing else. (a) What computational cost does this add per normalization
            call? (b) The paper's claim is that the two are similar in stability &mdash; so what is the argument for
            preferring RMSNorm, given they perform comparably? Tie it to the paper's theme.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Identify the extra work LayerNorm does: a pass to compute the mean $\mu$, then a variance around $\mu$, plus a learned shift $b$ to store and update. — _RMSNorm needs only $\sum x_i^2$ (one statistic) and only a gain; LayerNorm needs the mean and a centered variance and a shift._
- Note that this extra work runs at every sub-layer, every token, every forward pass &mdash; including at inference. — _Normalization is called constantly; a small per-call saving is multiplied by an enormous number of calls._
- Connect to LLaMA's theme: the model is meant to be served cheaply, so even small per-call savings at inference are worth taking when quality is unchanged. — _The whole paper optimizes the inference budget; RMSNorm is a small instance of that same principle._

**Answer:** (a) Putting LayerNorm back adds the mean-subtraction (an extra pass over the vector to compute
                 $\mu$ and a centered variance) plus a learned shift $b$ to store &mdash; more compute and one more
                 parameter vector per norm. (b) Since the paper reports the two are comparably stable, RMSNorm wins
                 on cost: it is slightly cheaper at every sub-layer, every token, every forward pass,
                 including inference. That fits LLaMA's central theme &mdash; optimize the inference budget &mdash;
                 so the cheaper option is preferred when quality is unchanged.

</details>